# Maia Fine-Tuning - Full Luxus O.S Knowledge
This notebook trains the Maia model on the entire Luxus O.S repository, including:
- 🤖 **Agents**: Architectures and roles
- 🧠 **Skills**: 36,000+ system capabilities
- 🔄 **Workflows**: Process automations
- 🛠️ **Plugins & Connectors**: MCP and core extensions
- 📝 **Training Prompts**: 44,000+ Q&A pairs

Powered by Unsloth.

In [ ]:
%%capture
%pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel # type: ignore
import torch # type: ignore
max_seq_length = 262144
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained( # type: ignore
    model_name = "google/gemma-4-E4B-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
!wget https://github.com/calitosaa/Luxus-O.S/raw/main/Maia/training_data.zip
!unzip -o training_data.zip

from datasets import load_dataset # type: ignore
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = f"Below is an instruction that describes a task related to Luxus O.S. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Context:\n{input}\n\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer # type: ignore
from transformers import TrainingArguments # type: ignore

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        max_steps = 1000, # Increased for deep repo knowledge
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
from google.colab import files # type: ignore
files.download("model/unsloth.Q4_K_M.gguf")